In [1]:
"""
VIX Volatility Sampler (Simple)
================================
Downloads historical VIX data via yfinance and samples random volatilities
by picking random indices from the historical series.
"""

import numpy as np
import pandas as pd
import yfinance as yf


class VIXVolatilitySampler:
    def __init__(self, period: str = "10y"):
        """
        Downloads VIX historical closing prices.

        Args:
            period (str): yfinance period string (e.g., '5y', '10y', 'max')
        """
        data = yf.download("^VIX", period=period, auto_adjust=True, progress=False)

        if data.empty:
            raise RuntimeError("Failed to download VIX data.")

        # Handle multi-level columns from newer yfinance
        if isinstance(data.columns, pd.MultiIndex):
            data.columns = data.columns.get_level_values(0)

        # Store closing prices as decimal volatilities (e.g., 20% -> 0.20)
        self.vix_close = data["Close"].dropna().values.flatten()
        self.vix_decimal = self.vix_close / 100.0
        self.n_obs = len(self.vix_decimal)

        print(f"Loaded {self.n_obs} VIX observations")
        print(f"VIX range: {self.vix_close.min():.2f}% – {self.vix_close.max():.2f}%")

    def sample(self, n: int = 1) -> np.ndarray:
        """
        Draw n random volatilities by picking random indices from historical VIX.

        Args:
            n (int): Number of samples to draw.

        Returns:
            np.ndarray: Annualized volatilities in decimal form (e.g., 0.18 = 18%).
        """
        random_indices = np.random.randint(0, self.n_obs, size=n)
        return self.vix_decimal[random_indices]


# ====================================================================== #
if __name__ == "__main__":
    sampler = VIXVolatilitySampler(period="10y")

    # Draw 10 random volatilities
    sigmas = sampler.sample(n=10)
    print(f"\n10 sampled volatilities (decimal): {sigmas.round(4)}")
    print(f"As VIX levels (%):                 {(sigmas * 100).round(2)}")

    # Draw 50,000 for Monte Carlo
    mc_sigmas = sampler.sample(n=50_000)
    print(f"\n50k samples — mean: {mc_sigmas.mean():.4f}, std: {mc_sigmas.std():.4f}")

Loaded 2515 VIX observations
VIX range: 9.14% – 82.69%

10 sampled volatilities (decimal): [0.1615 0.1262 0.1137 0.1016 0.207  0.264  0.1946 0.1363 0.2667 0.1281]
As VIX levels (%):                 [16.15 12.62 11.37 10.16 20.7  26.4  19.46 13.63 26.67 12.81]

50k samples — mean: 0.1847, std: 0.0733
